# Validate the FD002 and FD003 input files.

Recorded-run workflow. Configure paths in `settings.json`; see the root README for dependencies.


In [ ]:
from pathlib import Path
import sys

_release = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "release_runtime.py").is_file()), None)
if _release is None:
    raise FileNotFoundError("Open this notebook from the extracted Code_Release_Candidate folder.")
sys.path.insert(0, str(_release))
from release_runtime import activate_project, load_json, load_text
PROJECT_ROOT, RELEASE_CLASSICAL_PYTHON, RELEASE_TORCH_PYTHON = activate_project()


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib,json,platform
import numpy as np
import pandas as pd


DATA_DIR=Path("data")
OUTPUT_DIR=Path("revision_outputs")/"08_FD002_FD003_data_audit"
DATASETS=("FD002","FD003")
EXPECTED_UNITS={"FD002":{"train":260,"test":259},"FD003":{"train":100,"test":100}}
DECLARED={"FD002":{"operating_conditions":6,"failure_modes":1},
          "FD003":{"operating_conditions":1,"failure_modes":2}}
SENSORS=[f"Sensor{i}" for i in range(1,22)]
SETTINGS=["OpSet1","OpSet2","OpSet3"]
IDS=["UnitNumber","Time"]
COLUMNS=IDS+SETTINGS+SENSORS
SOURCE_URL="https://data.nasa.gov/dataset/cmapss-jet-engine-simulated-data"
FD001_SHA256={"train":"963b5e22825b34d8b21c69e1aeb4af3e647050eb672ee8834ba4b5d91d2de0f8",
              "test":"3cda7109ce17bafb5443f2ac926cfcf88154b941b8c4cf95eb55d1ddd6f52851"}
print("Veri klasörü:",DATA_DIR.resolve())


In [ ]:
def require(ok,message):
    if not bool(ok):raise ValueError(message)

def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def read_engine_file(path,expected_units,read_files=None):
    raw=pd.read_csv(path,sep=r"\s+",header=None)
    if read_files is not None:read_files.append(path.name)
    require(raw.shape[1]==26,f"{path.name}: 26 sütun bekleniyor; bulunan {raw.shape[1]}.")
    raw.columns=COLUMNS
    values=raw.to_numpy(dtype=np.float64)
    require(np.isfinite(values).all(),f"{path.name}: eksik/sonsuz değer var.")
    for name in IDS:
        a=raw[name].to_numpy(dtype=np.float64)
        require(np.all(a==np.floor(a)) and np.all(a>0),f"{path.name}: {name} pozitif tam sayı olmalı.")
        raw[name]=raw[name].astype('int64')
    units=np.sort(raw.UnitNumber.unique())
    require(np.array_equal(units,np.arange(1,expected_units+1)),
            f"{path.name}: motor kimlikleri 1..{expected_units} olmalı; bulunan motor sayısı {len(units)}.")
    require(not raw.duplicated(IDS).any(),f"{path.name}: tekrarlanan motor/çevrim var.")
    for unit,g in raw.groupby('UnitNumber',sort=False):
        require(np.array_equal(g.Time.to_numpy(),np.arange(1,len(g)+1)),
                f"{path.name}: motor {unit} çevrimleri 1'den başlayıp sıralı ilerlemiyor.")
    lengths=raw.groupby('UnitNumber').size()
    info={"shape":[int(raw.shape[0]),26],"units":int(len(units)),"sensor_count":21,
          "nonfinite_values":0,"duplicate_engine_cycle_pairs":0,
          "cycles_start_at_one_and_are_consecutive":True,
          "cycles_per_engine":{"min":int(lengths.min()),"median":float(lengths.median()),"max":int(lengths.max())}}
    return raw,info

def read_rul_file(path,expected_units,read_files=None):
    r=pd.read_csv(path,sep=r"\s+",header=None)
    if read_files is not None:read_files.append(path.name)
    require(r.shape==(expected_units,1),f"{path.name}: {expected_units} satır ve 1 sütun bekleniyor; bulunan {r.shape}.")
    a=r.iloc[:,0].to_numpy(dtype=np.float64)
    require(np.isfinite(a).all() and np.all(a>=0) and np.all(a==np.floor(a)),
            f"{path.name}: RUL değerleri sonlu, negatif olmayan tam sayılar olmalı.")
    return pd.Series(a.astype('int64'),index=np.arange(1,expected_units+1),name='terminal_RUL')

def build_metadata(frame,dataset,split,terminal_rul=None):
    m=frame[IDS].copy()
    m.insert(0,'source_row',np.arange(len(frame),dtype='int64'))
    m.insert(0,'split',split)
    m.insert(0,'dataset',dataset)
    m['engine_key']=dataset+':'+split+':'+m.UnitNumber.astype(str)
    m['sample_key']=m.engine_key+':'+m.Time.astype(str)
    rul=frame.groupby('UnitNumber').Time.transform('max')-frame.Time
    if split=='test':
        require(terminal_rul is not None,'Test son çevrim RUL vektörü yok.')
        offset=frame.UnitNumber.map(terminal_rul)
        require(offset.notna().all(),'Test motoru RUL vektörüyle eşleşmedi.')
        rul=rul+offset
    m['RUL']=rul.to_numpy(dtype='int64')
    require(m.sample_key.is_unique and (m.RUL>=0).all(),'Örnek anahtarı veya RUL hatası.')
    for _,g in m.groupby('engine_key',sort=False):
        require(np.all(np.diff(g.RUL.to_numpy())==-1),'RUL çevrim başına bir azalmıyor.')
    last=m.groupby('UnitNumber',sort=True).tail(1).set_index('UnitNumber').RUL.sort_index()
    if split=='train':require(last.eq(0).all(),'Eğitim son çevrim RUL değerleri sıfır değil.')
    else:require(np.array_equal(last.to_numpy(),terminal_rul.sort_index().to_numpy()),'Resmî son çevrim RUL eşleşmesi hatalı.')
    return m

def audit_dataset(dataset,data_dir,out_dir,expected_units,read_files=None):
    paths={split:data_dir/f'{split}_{dataset}.txt' for split in ['train','test']}
    paths['rul']=data_dir/f'RUL_{dataset}.txt'
    hashes={p.name:digest(p) for p in paths.values()}
    for split in ['train','test']:
        require(hashes[paths[split].name]!=FD001_SHA256[split],
                f'{paths[split].name}: daha önce denetlenen FD001 dosyasıyla aynı; dosya adını/alt kümeyi kontrol edin.')
    train,train_info=read_engine_file(paths['train'],expected_units['train'],read_files)
    test,test_info=read_engine_file(paths['test'],expected_units['test'],read_files)
    terminal=read_rul_file(paths['rul'],expected_units['test'],read_files)
    train_meta=build_metadata(train,dataset,'train')
    test_meta=build_metadata(test,dataset,'test',terminal)
    require(set(train_meta.engine_key).isdisjoint(set(test_meta.engine_key)),'Train/test motor anahtarları çakışıyor.')
    refs=[]
    for w in [20,30,40]:
        y=train_meta.RUL.le(w)
        lengths=train.groupby('UnitNumber').size()
        refs.append({'RUL_window':w,'positive':int(y.sum()),'negative':int((~y).sum()),
                     'prevalence':float(y.mean()),'engines_with_at_most_window_cycles':int((lengths<=w).sum())})
    sensor_audit=pd.DataFrame([{'sensor':s,'train_unique_values':int(train[s].nunique()),
                 'train_mean':float(train[s].mean()),'train_std':float(train[s].std(ddof=0)),
                 'train_min':float(train[s].min()),'train_max':float(train[s].max()),
                 'constant_in_train':bool(train[s].nunique()==1)} for s in SENSORS])
    settings=[{'setting':s,'train_unique_values':int(train[s].nunique()),
               'train_min':float(train[s].min()),'train_max':float(train[s].max()),
               'train_mean':float(train[s].mean()),'train_std':float(train[s].std(ddof=0))} for s in SETTINGS]
    for p in paths.values():require(digest(p)==hashes[p.name],f'Okuma sırasında dosya değişmiş: {p.name}')
    out_dir.mkdir(parents=True,exist_ok=True)
    train_meta.to_csv(out_dir/'train_evaluation_metadata.csv',index=False)
    test_meta.to_csv(out_dir/'test_evaluation_metadata.csv',index=False)
    sensor_audit.to_csv(out_dir/'sensor_audit.csv',index=False)
    pd.DataFrame(settings).to_csv(out_dir/'training_settings_audit.csv',index=False)
    output_names=['train_evaluation_metadata.csv','test_evaluation_metadata.csv','sensor_audit.csv','training_settings_audit.csv']
    return {'dataset':dataset,'status':'PASS','train':train_info,'test':test_info,
            'declared_dataset_properties':DECLARED[dataset],
            'source_raw_file_sha256':hashes,'feature_columns':SENSORS,
            'metadata_columns':list(train_meta.columns),'training_reference_counts':refs,
            'constant_training_sensors':sensor_audit.loc[sensor_audit.constant_in_train,'sensor'].tolist(),
            'training_settings':settings,
            'RUL_check':{'official_test_terminal_values':int(len(terminal)),
                         'test_terminal_matches':int(expected_units['test']),
                         'all_train_terminal_RUL_zero':True,'decreases_by_one_each_cycle':True},
            'sensors_removed':[],'rows_removed':0,
            'output_file_sha256':{dataset+'/'+name:digest(out_dir/name) for name in output_names}}

def run_audit(data_dir,output_dir):
    output_dir.mkdir(parents=True,exist_ok=True)
    read_files=[]
    summary={'stage':'08_additional_data_audit','status':'PENDING','datasets':{},
             'versions':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__},
             'created_at_utc':datetime.now(timezone.utc).isoformat(),
             'source_description_url':SOURCE_URL,'model_training_performed':False,
             'test_files_read':False,'test_read_purpose':'schema and terminal RUL alignment only; no model evaluation',
             'performance_metrics_computed':False,'learned_preprocessing_performed':False,
             'RUL_in_model_inputs':False,'reference_note':'RUL-derived late-life proxy is not anomaly-onset ground truth',
             'manuscript_author_confirmed_complete':['M01','M02','M03','M04','M05','section_4.2','Figure_5']}
    missing=[str(data_dir/name) for ds in DATASETS for name in [f'train_{ds}.txt',f'test_{ds}.txt',f'RUL_{ds}.txt']
             if not (data_dir/name).is_file()]
    if missing:
        summary.update(status='MISSING_FILES',missing_files=missing)
        print('Eksik dosyalar:')
        for p in missing:print(' ',p)
    else:
        errors=[]
        for ds in DATASETS:
            try:
                result=audit_dataset(ds,data_dir,output_dir/ds,EXPECTED_UNITS[ds],read_files)
                summary['datasets'][ds]=result
                print(ds,'PASS | train:',result['train']['shape'],'test:',result['test']['shape'],flush=True)
            except (ValueError,OSError,KeyError,TypeError) as exc:
                errors.append({'dataset':ds,'error_type':type(exc).__name__,'message':str(exc)})
                summary['datasets'][ds]={'dataset':ds,'status':'FAIL','error':str(exc)}
                print(ds,'FAIL:',exc,flush=True)
        summary['status']='PASS' if not errors else 'FAIL'
        if errors:summary['errors']=errors
    summary['files_read_successfully']=read_files
    summary['test_files_read']=any(name.startswith('test_') for name in read_files)
    out=output_dir/'additional_data_summary.json'
    out.write_text(json.dumps(summary,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
    print('ADDITIONAL DATA AUDIT:',summary['status'])
    print('Gönderilecek dosya:',out.resolve())
    return summary


In [ ]:
additional_data_summary=run_audit(DATA_DIR,OUTPUT_DIR)
